# Notebook 16b -- SDK Reference: Pulse, Operators, QASM, and Backends

**Prerequisites:** Notebooks 01-11. Familiarity with gates, noise, and circuit construction.

Notebook 16 covered the algorithmic frontier: Hamiltonian simulation,
HHL, Clifford simulation, and Pauli algebra. This companion notebook
completes the curriculum by surveying the **SDK-level capabilities**
that connect algorithms to real hardware and production workflows:
**pulse-level programming**, **operator representations**, **QASM/Quil
interop**, **backends**, and **observability hooks**.

By the end of this notebook you will be able to:

1. **Program** pulse-level waveforms below the gate abstraction.
2. **Convert** between operator representations (Kraus, SuperOp, Choi, PTM).
3. **Emit and parse** circuits in OpenQASM 3.0 and Quil formats.
4. **Submit** circuits to local, mock, and cloud backends through a unified interface.
5. **Attach** observability hooks for production-grade monitoring.

### What we will cover

| Topic | Key idea |
|-------|----------|
| Pulse-level control | Program waveforms below the gate abstraction |
| Operator representations | Kraus, SuperOp, Choi, PTM -- and conversions |
| QASM / Quil interop | Emit and parse standard circuit formats |
| Backends | Local simulator, mock backend, cloud targets |
| Observability | Hook into simulation and transpilation events |

In [1]:
import (
	"context"
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/backend"
	"github.com/splch/goqu/backend/local"
	"github.com/splch/goqu/backend/mock"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/observe"
	"github.com/splch/goqu/pulse"
	"github.com/splch/goqu/pulse/waveform"
	qasmEmitter "github.com/splch/goqu/qasm/emitter"
	"github.com/splch/goqu/qasm/parser"
	quilEmitter "github.com/splch/goqu/quil/emitter"
	"github.com/splch/goqu/sim/noise"
	"github.com/splch/goqu/sim/operator"
)

## Pulse-Level Programming

Gates are an abstraction. At the hardware level, quantum operations are
implemented by shaped microwave, laser, or RF **pulses** applied to
control lines (ports). Goqu's pulse package lets you program at this
lower level:

- **Port** -- a hardware I/O endpoint with a time resolution $dt$.
- **Frame** -- a software reference clock attached to a port, with a
  carrier frequency and phase.
- **Waveform** -- the pulse envelope (Gaussian, DRAG, constant, etc.).
- **Instructions** -- Play, Delay, ShiftPhase, SetFrequency, Capture, etc.

This is analogous to OpenPulse in Qiskit or calibration-level access
on IBM hardware.

In [2]:
%%
// Define a drive port with dt = 0.2222 ns (typical for IBM hardware).
port := pulse.MustPort("d0", 2.222e-10)
frame := pulse.MustFrame("d0_frame", port, 5.0e9, 0.0)

// Build a Gaussian X pulse followed by a phase-shifted pulse.
gaussian := waveform.MustGaussian(0.5, 160e-9, 40e-9)

prog, err := pulse.NewBuilder("x-pulse").
	AddPort(port).
	AddFrame(frame).
	Play(frame, gaussian).
	Delay(frame, 100e-9).
	ShiftPhase(frame, math.Pi/2).
	Play(frame, gaussian).
	Capture(frame, 1000e-9).
	Build()

if err != nil {
	fmt.Println("Error:", err)
} else {
	stats := prog.Stats()
	fmt.Printf("Pulse program: %d ports, %d frames, %d instructions\n",
		stats.NumPorts, stats.NumFrames, stats.NumInstructions)
	fmt.Printf("Total duration: %.2f ns\n", stats.TotalDuration*1e9)
	fmt.Println(prog)
}

Pulse program: 1 ports, 1 frames, 5 instructions
Total duration: 1420.00 ns
Program(x-pulse: 1 ports, 1 frames, 5 instructions, 1.42e-06 s)


## Operator Representations

A quantum channel can be described in four equivalent ways, each useful
for different purposes:

| Representation | Definition | Use case |
|:--|:--|:--|
| **Kraus** | $\mathcal{E}(\rho) = \sum_k E_k \rho E_k^\dagger$ | Simulation, physical intuition |
| **SuperOp** | $\text{vec}(\mathcal{E}(\rho)) = S \cdot \text{vec}(\rho)$ | Composition (matrix multiply) |
| **Choi** | $\Lambda = (\mathcal{I} \otimes \mathcal{E})(|\Omega\rangle\langle\Omega|)$ | CP checks (positive semidefinite?) |
| **PTM** | $R_{ij} = \text{Tr}(\sigma_i \cdot \mathcal{E}(\sigma_j)) / d$ | Pauli error analysis |

Goqu provides conversion functions between all representations.

In [3]:
%%
// Start from a depolarizing noise channel and convert between representations.
ch := noise.Depolarizing1Q(0.1)
kraus := operator.FromChannel(ch)

superOp := operator.KrausToSuperOp(kraus)
choi := operator.KrausToChoi(kraus)
ptm := operator.KrausToPTM(kraus)

fmt.Printf("Depolarizing(p=0.1) representations:\n")
fmt.Printf("  Kraus operators:    %d\n", len(kraus.Operators()))
fmt.Printf("  SuperOp matrix:     %d elements (4x4)\n", len(superOp.Matrix()))
fmt.Printf("  Choi matrix:        %d elements (4x4)\n", len(choi.Matrix()))
fmt.Printf("  PTM matrix:         %d elements (4x4)\n", len(ptm.Matrix()))

// Fidelity measures.
fmt.Printf("\nFidelity measures:\n")
fmt.Printf("  Process fidelity:      %.4f\n", operator.ProcessFidelity(kraus))
fmt.Printf("  Average gate fidelity: %.4f\n", operator.AverageGateFidelity(kraus))

// CPTP validation.
fmt.Printf("\nPhysicality check:\n")
fmt.Printf("  Is CPTP: %v\n", operator.IsCPTP(kraus, 1e-10))

// Round-trip: Kraus -> Choi -> Kraus and verify fidelity is preserved.
roundTrip := operator.ChoiToKraus(choi)
fmt.Printf("\nRound-trip (Kraus -> Choi -> Kraus):\n")
fmt.Printf("  Original process fidelity:    %.6f\n", operator.ProcessFidelity(kraus))
fmt.Printf("  Round-trip process fidelity:  %.6f\n", operator.ProcessFidelity(roundTrip))

Depolarizing(p=0.1) representations:
  Kraus operators:    4
  SuperOp matrix:     16 elements (4x4)
  Choi matrix:        16 elements (4x4)
  PTM matrix:         16 elements (4x4)

Fidelity measures:
  Process fidelity:      0.9000
  Average gate fidelity: 0.9333

Physicality check:
  Is CPTP: true

Round-trip (Kraus -> Choi -> Kraus):
  Original process fidelity:    0.900000
  Round-trip process fidelity:  0.900000


## QASM and Quil Interop

Quantum circuits need to be exchanged between tools, compilers, and
hardware vendors. Goqu supports two standard formats:

- **OpenQASM 3.0** -- the IBM-originated standard used across the
  industry. `qasmEmitter.EmitString` produces QASM, and
  `parser.ParseString` reads it back.

- **Quil** -- Rigetti's instruction set. `quilEmitter.EmitString`
  produces Quil source.

Let's round-trip a circuit through OpenQASM 3.0 and also emit Quil.

In [4]:
%%
c, _ := builder.New("qasm-demo", 2).H(0).CNOT(0, 1).MeasureAll().Build()

fmt.Println("Original circuit:")
gonbui.DisplayHTML(draw.SVG(c))

// Emit to OpenQASM 3.0.
qasmStr, err := qasmEmitter.EmitString(c)
if err != nil {
	fmt.Println("QASM emit error:", err)
} else {
	fmt.Println("OpenQASM 3.0:")
	fmt.Println(qasmStr)
}

// Parse back from QASM.
parsed, err := parser.ParseString(qasmStr)
if err != nil {
	fmt.Println("Parse error:", err)
} else {
	fmt.Println("Parsed back from QASM:")
	gonbui.DisplayHTML(draw.SVG(parsed))
}

Original circuit:
OpenQASM 3.0:
OPENQASM 3.0;
include "stdgates.inc";
qubit[2] q;
bit[2] c;

h q[0];
cx q[0], q[1];
c[0] = measure q[0];
c[1] = measure q[1];

Parsed back from QASM:


q0 
 
 q1 
 
 H 
 
 
 
 M 
 M

q0 
 
 q1 
 
 H 
 
 
 
 M 
 M

In [5]:
%%
// Emit the same circuit to Quil (Rigetti format).
c, _ := builder.New("quil-demo", 2).H(0).CNOT(0, 1).MeasureAll().Build()

quilStr, err := quilEmitter.EmitString(c)
if err != nil {
	fmt.Println("Quil emit error:", err)
} else {
	fmt.Println("Quil:")
	fmt.Println(quilStr)
}

Quil:
DECLARE ro BIT[2]

H 0
CNOT 0 1
MEASURE 0 ro[0]
MEASURE 1 ro[1]



## Backends

The `backend.Backend` interface abstracts over execution targets:

```go
type Backend interface {
    Name() string
    Target() target.Target
    Submit(ctx, req) (*Job, error)
    Status(ctx, jobID) (*JobStatus, error)
    Result(ctx, jobID) (*Result, error)
    Cancel(ctx, jobID) error
}
```

Goqu ships with:

- **`local.Backend`** -- runs circuits on the local statevector simulator.
- **`mock.Backend`** -- a configurable test double for job pipelines.
- Cloud backends for IBM, Google, IonQ, Quantinuum, Rigetti, and AWS Braket.

All backends share the same submit/status/result lifecycle, so code
written against one backend works with any other.

In [6]:
%%
// Local backend: runs circuits on the local statevector simulator.
lb := local.New()
fmt.Printf("Local backend: %q\n", lb.Name())

c, _ := builder.New("bell", 2).H(0).CNOT(0, 1).MeasureAll().Build()
ctx := context.Background()

job, err := lb.Submit(ctx, &backend.SubmitRequest{
	Circuit: c,
	Shots:   1000,
	Name:    "bell-test",
})
if err != nil {
	fmt.Println("Submit error:", err)
} else {
	fmt.Printf("Job submitted: ID=%s, State=%s\n", job.ID, job.State)

	res, err := lb.Result(ctx, job.ID)
	if err != nil {
		fmt.Println("Result error:", err)
	} else {
		fmt.Printf("Results (%d shots):\n", res.Shots)
		for bitstring, count := range res.ToCounts() {
			fmt.Printf("  |%s>: %d\n", bitstring, count)
		}
	}
}

// Mock backend: configurable test double.
mb := mock.New("test-backend")
fmt.Printf("\nMock backend: %q\n", mb.Name())

Local backend: "local.simulator"


2026/03/16 17:35:59 INFO simulating circuit qubits=2 gates=4 shots=1000
2026/03/16 17:35:59 INFO simulation complete qubits=2 elapsed=49.375µs


Job submitted: ID=4986070fc66f022162d68bc91787057f, State=completed
Results (1000 shots):
  |00>: 511
  |11>: 489

Mock backend: "test-backend"


## Observability Hooks

When running quantum workloads in production, you need visibility into
what is happening: how long simulations take, how deep transpiled
circuits are, how many backend calls are made. Goqu's `observe` package
provides **zero-dependency hooks** that you attach to a `context.Context`.

Available hooks:

- `WrapSim` -- called around simulation execution.
- `WrapTranspile` -- called around the transpilation pipeline.
- `WrapPass` -- called around each individual transpilation pass.
- `WrapJob` -- called around job submission through result retrieval.
- `WrapHTTP` -- called around backend HTTP requests.
- `WrapSweep` -- called around parameter sweep execution.
- `OnJobPoll` -- called each time a job is polled for status.

This integrates naturally with OpenTelemetry or Prometheus via the
`otelbridge` and `prombridge` sub-packages.

In [7]:
%%
hooks := &observe.Hooks{
	WrapSim: func(ctx context.Context, info observe.SimInfo) (context.Context, func(err error)) {
		fmt.Printf("[observe] Simulating %d-qubit circuit (%d gates, %d shots)...\n",
			info.NumQubits, info.GateCount, info.Shots)
		return ctx, func(err error) {
			if err == nil {
				fmt.Println("[observe] Simulation complete!")
			} else {
				fmt.Printf("[observe] Simulation failed: %v\n", err)
			}
		}
	},
}

ctx := observe.WithHooks(context.Background(), hooks)

// Now any simulation run with this context will trigger the hook.
// The local backend respects observe hooks.
lb := local.New()
c, _ := builder.New("observed", 2).H(0).CNOT(0, 1).MeasureAll().Build()
job, err := lb.Submit(ctx, &backend.SubmitRequest{Circuit: c, Shots: 100})
if err != nil {
	fmt.Println("Error:", err)
} else {
	res, _ := lb.Result(ctx, job.ID)
	fmt.Printf("\nGot %d shots with hooks active.\n", res.Shots)
}

fmt.Println("\nHooks are zero-cost when not attached (nil check).")
fmt.Println("In production, wire them to OpenTelemetry for distributed tracing.")

[observe] Simulating 2-qubit circuit (4 gates, 100 shots)...
[observe] Simulation complete!

Got 100 shots with hooks active.

Hooks are zero-cost when not attached (nil check).
In production, wire them to OpenTelemetry for distributed tracing.


2026/03/16 17:36:00 INFO simulating circuit qubits=2 gates=4 shots=100
2026/03/16 17:36:00 INFO simulation complete qubits=2 elapsed=27.667µs


## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. Why is a virtual Z-rotation "free" in pulse-level programming while a physical Y-rotation costs gate time?
2. What property must a set of Kraus operators satisfy to represent a valid (physical) quantum channel?
3. Why is the QASM round-trip test important for verifying that circuit semantics are preserved across tools?

---

## Exercises

### Exercise 1 -- Custom Pulse Sequence for a Hadamard Gate

On superconducting hardware, the Hadamard gate $H = (X + Z)/\sqrt{2}$
can be decomposed as $H = RZ(\pi) \cdot RY(\pi/2)$. Build a pulse
sequence that implements this: a Y-rotation pulse (Gaussian on the
drive frame) followed by a Z-rotation (a virtual phase shift, which is
free on hardware).

In [8]:
%%
// Exercise 1: Custom Pulse Sequence for a Hadamard Gate
//
// Goal: Build a pulse program that implements H = RZ(pi) . RY(pi/2).
//       RY is a physical Gaussian pulse; RZ is a virtual phase shift.
//
// TODO: Create a drive port and frame
// TODO: Build the pulse program with Play (for RY) and ShiftPhase (for RZ)
//
// Skeleton:

// Step 1: Define hardware port and frame
// port := pulse.MustPort("d0", 2.222e-10)
// frame := pulse.MustFrame("d0_frame", port, 5.0e9, 0.0)

// Step 2: Create the RY(pi/2) Gaussian pulse
// The amplitude is calibrated so the pulse area equals pi/2.
// ryPulse := waveform.MustGaussian(???, 160e-9, 40e-9)

// Step 3: Build the pulse program
// hadamardProg, err := pulse.NewBuilder("hadamard-pulse").
//     AddPort(???).
//     AddFrame(???).
//     Play(frame, ???).           // RY(pi/2) via physical pulse
//     ShiftPhase(frame, ???).     // RZ(pi) via virtual phase shift
//     Build()
//
// if err != nil {
//     fmt.Println("Error:", err)
// } else {
//     stats := hadamardProg.Stats()
//     fmt.Printf("Hadamard pulse program:\n")
//     fmt.Printf("  Instructions: %d\n", stats.NumInstructions)
//     fmt.Printf("  Total duration: %.1f ns\n", stats.TotalDuration*1e9)
//     fmt.Println(hadamardProg)
// }

fmt.Println("TODO: Uncomment the code above, fill in the ???, and run!")

TODO: Uncomment the code above, fill in the ???, and run!


### Exercise 2 -- QASM Round-Trip with a 3-Qubit Circuit

Build a 3-qubit GHZ circuit ($H(0) \to CNOT(0,1) \to CNOT(1,2)$),
emit it to OpenQASM 3.0, parse it back, and verify that the parsed
circuit has the same gate count and qubit count as the original.

In [9]:
%%
// Exercise 2: QASM Round-Trip with a 3-Qubit Circuit
//
// Goal: Build a 3-qubit GHZ circuit, emit to QASM, parse back,
//       and verify structural equivalence.
//
// TODO: Build the GHZ circuit
// TODO: Emit to QASM and parse back
// TODO: Compare gate counts
//
// Skeleton:

// Step 1: Build the circuit
// c, _ := builder.New("ghz3", 3).H(0).CNOT(0, 1).CNOT(1, 2).MeasureAll().Build()

// Step 2: Emit to QASM
// qasmStr, err := qasmEmitter.EmitString(c)
// if err != nil {
//     fmt.Println("Emit error:", err)
// }
// fmt.Println("QASM output:")
// fmt.Println(qasmStr)

// Step 3: Parse back and compare
// parsed, err := parser.ParseString(qasmStr)
// if err != nil {
//     fmt.Println("Parse error:", err)
// }
// fmt.Printf("Original:  %d qubits, %d gates\n", c.NumQubits(), c.Stats().GateCount)
// fmt.Printf("Parsed:    %d qubits, %d gates\n", parsed.NumQubits(), parsed.Stats().GateCount)

fmt.Println("TODO: Uncomment the code above and run!")

TODO: Uncomment the code above and run!


### Exercise 3 -- Backend Job Pipeline

Submit a 3-qubit circuit to the local backend and the mock backend.
Compare the results. The mock backend returns uniform random counts,
while the local backend produces physically correct distributions.

In [10]:
%%
// Exercise 3: Backend Job Pipeline
//
// Goal: Submit the same circuit to local and mock backends,
//       compare the result distributions.
//
// TODO: Build a 3-qubit GHZ circuit
// TODO: Submit to local backend and collect results
// TODO: Submit to mock backend and collect results
// TODO: Print both distributions side by side
//
// Skeleton:

// ctx := context.Background()
// c, _ := builder.New("ghz3", 3).H(0).CNOT(0, 1).CNOT(1, 2).MeasureAll().Build()

// Step 1: Local backend
// lb := local.New()
// job1, _ := lb.Submit(ctx, &backend.SubmitRequest{Circuit: c, Shots: 1000})
// res1, _ := lb.Result(ctx, job1.ID)
// fmt.Println("Local backend results:")
// for bs, count := range res1.ToCounts() {
//     fmt.Printf("  |%s>: %d\n", bs, count)
// }

// Step 2: Mock backend
// mb := mock.New("test")
// job2, _ := mb.Submit(ctx, &backend.SubmitRequest{Circuit: c, Shots: 1000})
// res2, _ := mb.Result(ctx, job2.ID)
// fmt.Println("\nMock backend results:")
// for bs, count := range res2.ToCounts() {
//     fmt.Printf("  |%s>: %d\n", bs, count)
// }

fmt.Println("TODO: Uncomment the code above and run!")

TODO: Uncomment the code above and run!


## Key Takeaways -- Curriculum Conclusion

Over sixteen notebooks (plus this SDK reference), we have built a
comprehensive understanding of quantum computing using the Goqu SDK.

From Notebook 16 (advanced algorithms):

1. **Hamiltonian simulation** via Trotter-Suzuki decomposition lets us
   approximate $e^{-iHt}$ for arbitrary Hamiltonians.
2. **HHL** solves linear systems $Ax = b$ by encoding the solution in
   quantum amplitudes.
3. **Clifford simulation** using stabilizer tableaux can handle up to
   100,000 qubits in polynomial time.
4. **Pauli algebra** provides the mathematical foundation for expressing
   Hamiltonians and computing expectation values.

From this notebook (SDK reference):

5. **Pulse-level programming** goes below the gate abstraction to
   control the actual microwave/laser signals. Virtual Z-rotations are
   free; physical pulses have finite duration and error.

6. **Operator representations** (Kraus, SuperOp, Choi, PTM) provide
   different views of the same quantum channel, each useful for different
   tasks: simulation, composition, physicality checks, and error analysis.

7. **QASM/Quil interop** enables circuit exchange across the quantum
   ecosystem. Round-trip parsing preserves circuit semantics.

8. **Backends** abstract over execution targets (local simulators, cloud
   hardware). The same code runs on any backend through the unified
   `Submit/Status/Result` interface.

9. **Observability hooks** provide production-grade visibility into
   quantum workloads without coupling to any specific metrics or tracing
   framework.

---

**Congratulations on completing the quantum computing curriculum!**
You now have the tools and understanding to build, simulate, transpile,
and run quantum algorithms -- from single-qubit gates all the way to
production-grade Hamiltonian simulation with observability.

The quantum advantage is not in any single algorithm, but in the ability
to compose all of these pieces: express a problem as a Hamiltonian,
simulate it with the right decomposition, transpile to hardware
constraints, mitigate noise, and monitor the entire pipeline.